# Part 1 - Multimodal RAG over a read-only knowledge base
Pipeline: PyMuPDF -> text chunks + figure crops -> Qwen2.5-VL descriptions -> ChromaDB + BM25 -> hybrid (RRF) retrieval -> grounded JSON answers with per-conversation memory.
Data: attach the Kaggle competition data (Document*.pdf, kb_manifest.json, Questions.csv) or set `DATA_DIR`.

In [ ]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

In [ ]:
import os, sys, json, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
pd.set_option("display.max_colwidth", 120)

## 1.1 Ingestion & indexing

In [ ]:
from p1_rag.pipeline import build, answer_file
from p1_rag.index import Retriever
from p1_rag.rag import RAGSystem
t0 = time.time()
final, items, figures = build(work_dir="work_p1", mode="hybrid")      # reranker_id="BAAI/bge-reranker-base" for the bonus
print("ingestion+index seconds:", round(time.time() - t0))
fdf = pd.DataFrame(figures)
print(len(items), "indexed items;", len(figures), "figures/tables;", int(fdf.untrusted.sum()), "inside Appendix C (excluded from retrieval)")
fdf[["doc_id", "page", "label", "kind", "trust_level", "untrusted", "cropped", "injection_in_image"]]

## 1.2 / 1.3 Retrieval, answering, conversations - sanity check

In [ ]:
chat = final.new_chat()
print(chat.answer("What does Figure 1-1 show?"))
print(final.last_retrieved[:5])
n_before = final.retriever.col.count()

## 1.4 Golden set (25 questions written by us from the PDFs, committed BEFORE tuning)
See `p1_rag/golden.py` for the schema. Save as `golden_set.json` in the repo root.

In [ ]:
from p1_rag import golden
G = golden.load("golden_set.json")
print("coverage problems:", golden.check_coverage(G) or "none", "| n =", len(G))

In [ ]:
# baseline = dense retrieval only, no follow-up rewriting; final = hybrid + conversation handling
baseline = RAGSystem(Retriever(items, figures, "work_p1/chroma_dense", mode="dense", embedder=final.retriever.embedder),
                     final.vlm, conversation=False, llm=final.llm)   # same answering model; only retrieval/conversation differ
rows = {}
for name, system in [("baseline", baseline), ("final", final)]:
    recs = golden.run_golden(system, G)
    rows[name] = golden.summarize(recs)
    pd.DataFrame(recs).to_csv(f"golden_results_{name}.csv", index=False)
results = pd.DataFrame(rows).T
results

In [ ]:
# Read-only knowledge base: answering must never change the vector store
assert final.retriever.col.count() == n_before, "vector store changed during answering!"
print("vector store unchanged:", n_before, "items")
golden.human_sheet(recs)   # (recs = final-system results from the loop above) both members rate 1-5 in human_eval_p1.csv, then:
# golden.human_summary("human_eval_p1.csv")

## C4 safety analysis - untrusted Appendix C, false claims, injections

In [ ]:
from p1_rag.text_utils import has_injection
untrusted = [i for i in items if i["untrusted"]]
print("Appendix C items kept out of retrieval:", len(untrusted))
print("figures whose VLM transcript contained instruction-like text:", [f["label"] for f in figures if f["injection_in_image"]])
print("text chunks (trusted) that read like injections before sanitising:", sum(has_injection(i["text"]) for i in items if not i["untrusted"]))

## 1.5 Kaggle submission (reads QUESTIONS_PATH, default Questions.csv; no manual steps)

In [ ]:
from p1_rag.ingest import find_data_dir
QUESTIONS_PATH = os.environ.get("QUESTIONS_PATH", os.path.join(find_data_dir(), "Questions.csv"))
sub = answer_file(final, QUESTIONS_PATH, "submission.csv")
sub.head()

In [ ]:
import subprocess
print(subprocess.run([sys.executable, os.path.join(find_data_dir(), "validate_submission.py"), "submission.csv", QUESTIONS_PATH], capture_output=True, text=True).stdout)